# Linear Probe Analysis

Probes the frozen intermediate representations of the classifier and regressor
to understand what each layer has learned.

**Setup:** Place your training data CSVs (`0716_dataset_train.csv`, `0716_dataset_valid.csv`,
`0716_dataset_test.csv`) in this directory, or update `DATA_DIR` below.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, mean_squared_error
import matplotlib.pyplot as plt

from qprimer_designer.models import (
    load_models, PcrDataset, FEATURE_COLUMNS, SEQUENCE_COLUMNS,
)
from qprimer_designer.utils import encode_batch_parallel

## 1. Load data and models

In [ ]:
DATA_DIR = "."
BATCH_SIZE = 256

train_df = pd.read_csv(f"{DATA_DIR}/0716_dataset_train.csv")
valid_df = pd.read_csv(f"{DATA_DIR}/0716_dataset_valid.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/0716_dataset_test.csv")

print(f"Train: {len(train_df)}, Valid: {len(valid_df)}, Test: {len(test_df)}")
train_df.head()

In [ ]:
scaler, classifier, regressor, device = load_models(device="cpu")
classifier.eval()
regressor.eval()
print(f"Device: {device}")
print(classifier)

## 2. Build dataloaders

In [ ]:
def make_loader(df, scaler, batch_size=BATCH_SIZE):
    features = scaler.transform(df[FEATURE_COLUMNS])
    sequences = encode_batch_parallel(df[SEQUENCE_COLUMNS])
    scores = df["score"].values if "score" in df.columns else np.zeros(len(df))
    dataset = PcrDataset(sequences, features, scores)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)

train_loader = make_loader(train_df, scaler)
valid_loader = make_loader(valid_df, scaler)
test_loader  = make_loader(test_df, scaler)

## 3. Extract intermediate representations

Register forward hooks on layers of interest, then run a forward pass
to collect activations.

In [ ]:
CLASSIFIER_LAYERS = {
    "mlp_hidden":   "mlp.model.0",    # Linear(12 -> 64), before ReLU
    "ssm_encoder":  "ssm.encoder",     # Linear(10 -> 32)
    "ssm_norm":     "ssm.norm",        # RMSNorm after PGC blocks (32-dim, seq_len)
    "combiner_hidden": "combiner.0",   # Linear(8 -> 32), before ReLU
}

REGRESSOR_LAYERS = {
    "mlp_hidden":   "mlp.model.0",    # Linear(12 -> 64)
    "dl_encoder":   "dl.encoder",      # Linear(10 -> 64)
    "dl_norm":      "dl.norm",         # RMSNorm after PGC blocks (64-dim, seq_len)
    "combiner_hidden": "combiner.0",   # Linear(8 -> 16)
}


def get_submodule(model, dotted_name):
    parts = dotted_name.split(".")
    mod = model
    for p in parts:
        mod = getattr(mod, p) if not p.isdigit() else mod[int(p)]
    return mod


def extract_representations(model, loader, layer_map, device="cpu"):
    """Run forward passes and collect layer outputs via hooks."""
    activations = {}
    hooks = []

    def make_hook(name):
        def hook_fn(module, inp, out):
            activations.setdefault(name, []).append(out.detach().cpu())
        return hook_fn

    for name, layer_path in layer_map.items():
        h = get_submodule(model, layer_path).register_forward_hook(make_hook(name))
        hooks.append(h)

    all_labels = []
    with torch.no_grad():
        for seq_in, feat_in, labels in loader:
            seq_in = seq_in.to(device).float()
            feat_in = feat_in.to(device).float()
            _ = model(feat_in, seq_in)
            all_labels.append(labels.numpy())

    for h in hooks:
        h.remove()

    result = {}
    for name, tensors in activations.items():
        combined = torch.cat(tensors, dim=0)
        # Mean-pool over sequence length if 3D (batch, seq_len, dim)
        if combined.ndim == 3:
            combined = combined.mean(dim=1)
        result[name] = combined.numpy()

    return result, np.concatenate(all_labels)

In [ ]:
print("Extracting classifier representations...")
cls_train, y_train = extract_representations(classifier, train_loader, CLASSIFIER_LAYERS)
cls_valid, y_valid = extract_representations(classifier, valid_loader, CLASSIFIER_LAYERS)
cls_test,  y_test  = extract_representations(classifier, test_loader,  CLASSIFIER_LAYERS)

# Binary labels for classification probes
y_train_bin = (y_train > 0).astype(int)
y_valid_bin = (y_valid > 0).astype(int)
y_test_bin  = (y_test > 0).astype(int)

for name, arr in cls_train.items():
    print(f"  {name}: {arr.shape}")

In [ ]:
print("Extracting regressor representations...")
reg_train, _ = extract_representations(regressor, train_loader, REGRESSOR_LAYERS)
reg_valid, _ = extract_representations(regressor, valid_loader, REGRESSOR_LAYERS)
reg_test,  _ = extract_representations(regressor, test_loader,  REGRESSOR_LAYERS)

for name, arr in reg_train.items():
    print(f"  {name}: {arr.shape}")

## 4. Classification probes

Fit logistic regression on frozen representations to predict `score > 0`.

In [ ]:
def run_classification_probe(X_train, X_test, y_train, y_test):
    probe = LogisticRegression(max_iter=2000, solver="lbfgs")
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)
    y_prob = probe.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "auroc": roc_auc_score(y_test, y_prob),
    }


print("=== Classification probes (classifier layers) ===")
cls_probe_results = {}
for name in CLASSIFIER_LAYERS:
    metrics = run_classification_probe(
        cls_train[name], cls_test[name], y_train_bin, y_test_bin
    )
    cls_probe_results[name] = metrics
    print(f"  {name:20s}  acc={metrics['accuracy']:.3f}  auroc={metrics['auroc']:.3f}")

print("\n=== Classification probes (regressor layers) ===")
reg_cls_probe_results = {}
for name in REGRESSOR_LAYERS:
    metrics = run_classification_probe(
        reg_train[name], reg_test[name], y_train_bin, y_test_bin
    )
    reg_cls_probe_results[name] = metrics
    print(f"  {name:20s}  acc={metrics['accuracy']:.3f}  auroc={metrics['auroc']:.3f}")

## 5. Regression probes

Fit Ridge regression on frozen representations to predict the raw `score`.

In [ ]:
def run_regression_probe(X_train, X_test, y_train, y_test):
    probe = Ridge(alpha=1.0)
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)
    return {
        "r2": r2_score(y_test, y_pred),
        "rmse": mean_squared_error(y_test, y_pred, squared=False),
    }


print("=== Regression probes (regressor layers) ===")
reg_probe_results = {}
for name in REGRESSOR_LAYERS:
    metrics = run_regression_probe(
        reg_train[name], reg_test[name], y_train, y_test
    )
    reg_probe_results[name] = metrics
    print(f"  {name:20s}  R²={metrics['r2']:.3f}  RMSE={metrics['rmse']:.3f}")

print("\n=== Regression probes (classifier layers) ===")
cls_reg_probe_results = {}
for name in CLASSIFIER_LAYERS:
    metrics = run_regression_probe(
        cls_train[name], cls_test[name], y_train, y_test
    )
    cls_reg_probe_results[name] = metrics
    print(f"  {name:20s}  R²={metrics['r2']:.3f}  RMSE={metrics['rmse']:.3f}")

## 6. Visualize results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Classification AUROC comparison
ax = axes[0]
all_cls = {}
for name, m in cls_probe_results.items():
    all_cls[f"cls/{name}"] = m["auroc"]
for name, m in reg_cls_probe_results.items():
    all_cls[f"reg/{name}"] = m["auroc"]

bars = ax.barh(list(all_cls.keys()), list(all_cls.values()))
ax.set_xlabel("AUROC")
ax.set_title("Classification Probe (score > 0)")
ax.set_xlim(0.5, 1.0)
for bar, v in zip(bars, all_cls.values()):
    ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center")

# Regression R² comparison
ax = axes[1]
all_reg = {}
for name, m in reg_probe_results.items():
    all_reg[f"reg/{name}"] = m["r2"]
for name, m in cls_reg_probe_results.items():
    all_reg[f"cls/{name}"] = m["r2"]

bars = ax.barh(list(all_reg.keys()), list(all_reg.values()))
ax.set_xlabel("R²")
ax.set_title("Regression Probe (raw score)")
for bar, v in zip(bars, all_reg.values()):
    ax.text(max(v + 0.01, 0.01), bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center")

plt.tight_layout()
plt.show()